# UF3: Visual Fidelity

Compares rendered DOM snapshots (bounding box, computed styles, text) of Ground
Truth vs. generated output. Snapshots are produced by `render_and_extract.py`
(Playwright) using the story manifest from `generate_eval_stories.mjs`.

Pipeline:
1. Element matching between GT and generated snapshot via bipartite (Hungarian)
   matching, cost based on PrimeVue component identity (`data-pc-name`), text
   similarity, and normalized center distance.
2. Per matched pair: IoU, position similarity, size similarity, style-match rate.
3. Per mockup: Element-Match-Rate (F1-style), aggregated over matched pairs.
4. Same aggregation/CSV-export structure as the other evaluation notebooks.

In [9]:
import re
import csv
import json
import difflib
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
from scipy.optimize import linear_sum_assignment

## 0. Repo root and configuration

In [10]:
def find_repo_root(marker: str = 'dataset', start: Path | None = None) -> Path:
    """Searches upward from the current working directory until a folder
    named 'marker' is found -- robust against the kernel's working directory
    not matching the notebook's own folder."""
    start = start or Path.cwd()

    for parent in [start, *start.parents]:
        if (parent / marker).is_dir():
            return parent

    raise FileNotFoundError(f"Could not find a folder named '{marker}' above {start}")


REPO_ROOT = find_repo_root()

# 'components' or 'uis'
TYPE = 'components'

SNAPSHOTS_DIR = REPO_ROOT / 'evaluations' / 'visual_fidelity_snapshots' / TYPE
COMPLEXITIES = ['simple', 'medium', 'hard']
VARIANTS = ['pretty', 'messy']

# Generic name for whichever second-level grouping applies for the current
# TYPE -- a complexity level for 'components', a pretty/messy variant for
# 'uis'. Used throughout instead of hardcoding COMPLEXITIES, so the same
# loading/aggregation code works for both directory layouts.
GROUP_LEVELS = COMPLEXITIES if TYPE == 'components' else VARIANTS
EXPECTED_GROUPS = set(GROUP_LEVELS)

# Approach 'a' is rule-based/deterministic and genuinely has NO prompt
# strategy -- not even a conceptual one, unlike B/C/D. It is still stored in
# the same 2-level {prompt_strategy}/{group} structure as B/C/D below, but
# under the real value None (not an invented label like 'deterministic'),
# consistent with storybook-generate-stories.ipynb / render_and_extract.
A_PROMPT_STRATEGY_KEY = None

# Placeholder 'group' key for GT under TYPE == 'uis', which has no
# complexity/variant subfolder at all (GT is variant-independent, just like
# the Figma JSON/screenshot it comes from).
GT_FLAT_GROUP_KEY = '_flat'

print(f'SNAPSHOTS_DIR: {SNAPSHOTS_DIR}')
print(f'Exists:        {SNAPSHOTS_DIR.exists()}')

if not SNAPSHOTS_DIR.exists():
    print('WARNING: Snapshots directory does not exist.')

# Style properties compared for Style-Match-Rate, split by comparison strategy.
COLOR_PROPS = {'color', 'backgroundColor', 'borderColor'}
LENGTH_PROPS = {'fontSize', 'padding', 'margin', 'borderRadius', 'gap', 'borderWidth'}
CATEGORICAL_PROPS = {'fontWeight', 'display', 'justifyContent', 'alignItems', 'textAlign'}
STYLE_PROPS = COLOR_PROPS | LENGTH_PROPS | CATEGORICAL_PROPS

# Matching cost weights (see match_cost()).
W_COMPONENT_MISMATCH = 1.0
W_TEXT_DISSIMILARITY = 0.5
W_POSITION_DISTANCE = 0.5
MAX_MATCH_COST = 1.2  # pairs above this cost are rejected even if Hungarian-optimal

SNAPSHOTS_DIR: C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\evaluations\visual_fidelity_snapshots\components
Exists:        True


## 1. Load snapshots (same directory convention as the other evaluation notebooks)

In [11]:
SNAPSHOTS_BY_APPROACH: dict = {
    'gt': defaultdict(dict),
    'a':  defaultdict(lambda: defaultdict(dict)),
    'b':  defaultdict(lambda: defaultdict(dict)),
    'c':  defaultdict(lambda: defaultdict(dict)),
    'd':  defaultdict(lambda: defaultdict(dict)),
}

APPROACHES = ['gt', 'a', 'b', 'c', 'd']

for approach in APPROACHES:
    approach_dir = SNAPSHOTS_DIR / approach

    print(f'Loading snapshots for approach {approach} from {approach_dir}')

    if not approach_dir.exists():
        print(f'    Skip (not existing): {approach_dir}')
        continue

    if approach == 'gt':
        if TYPE == 'components':
            for complexity_dir in approach_dir.iterdir():
                if not complexity_dir.is_dir() or complexity_dir.name not in EXPECTED_GROUPS:
                    continue

                complexity = complexity_dir.name

                for json_file in complexity_dir.glob('*.json'):
                    SNAPSHOTS_BY_APPROACH['gt'][complexity][json_file.stem] = json.loads(
                        json_file.read_text(encoding='utf-8'))

        else:
            # uis: gt/{1-5}.json -- flat, GT is variant-independent (same
            # snapshot used as reference for both pretty- and messy-derived generations).
            for json_file in approach_dir.glob('*.json'):
                SNAPSHOTS_BY_APPROACH['gt'][GT_FLAT_GROUP_KEY][json_file.stem] = json.loads(
                    json_file.read_text(encoding='utf-8'))

    elif approach == 'a':
        # components: a/{complexity}/{stem}.json
        # uis:        a/{variant}/{stem}.json
        # No prompt_strategy level either way -- stored under A_PROMPT_STRATEGY_KEY
        # (real None) so the rest of the notebook can treat 'a' uniformly
        # with b/c/d without special-casing it.
        for group_dir in approach_dir.iterdir():
            if not group_dir.is_dir() or group_dir.name not in EXPECTED_GROUPS:
                continue

            group = group_dir.name

            for json_file in group_dir.glob('*.json'):
                SNAPSHOTS_BY_APPROACH['a'][A_PROMPT_STRATEGY_KEY][group][json_file.stem] = json.loads(
                    json_file.read_text(encoding='utf-8'))

    else:
        for strategy_dir in approach_dir.iterdir():
            if not strategy_dir.is_dir():
                continue

            prompt_strategy = strategy_dir.name

            for group_dir in strategy_dir.iterdir():
                if not group_dir.is_dir() or group_dir.name not in EXPECTED_GROUPS:
                    continue

                group = group_dir.name

                for json_file in group_dir.glob('*.json'):
                    SNAPSHOTS_BY_APPROACH[approach][prompt_strategy][group][json_file.stem] = json.loads(
                        json_file.read_text(encoding='utf-8'))

print('\nSnapshots loaded:')
for approach, data in SNAPSHOTS_BY_APPROACH.items():
    if approach == 'gt':
        total = sum(len(files) for files in data.values())
        print(f'Approach {approach}: {total} snapshots')
    else:
        total = sum(len(files) for strategies in data.values() for files in strategies.values())
        print(f'Approach {approach}: {total} snapshots')
        for strategy, groups in data.items():
            strategy_total = sum(len(files) for files in groups.values())
            strategy_label = strategy if strategy is not None else '(none -- deterministic, approach a)'
            print(f'  Strategy {strategy_label}: {strategy_total} snapshots')

Loading snapshots for approach gt from C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\evaluations\visual_fidelity_snapshots\components\gt
Loading snapshots for approach a from C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\evaluations\visual_fidelity_snapshots\components\a
Loading snapshots for approach b from C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\evaluations\visual_fidelity_snapshots\components\b
Loading snapshots for approach c from C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\evaluations\visual_fidelity_snapshots\components\c
Loading snapshots for approach d from C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\evaluations\visual_fidelity_snapshots\components\d

Snapshots loaded:
Approach gt: 30 snapshots
Approach a: 30 snapshots
  Strategy (none -- deterministic, approach a): 30 snapshots
Approach b: 540 snapshots
  Strategy few_shot: 270 snapshots
  Strategy zero_shot: 270 snapshots
Approach c: 540 snapshots
  Strategy few_shot: 270 snapshots
  St

## 2. Style comparison

In [12]:
_COLOR_RE = re.compile(r'rgba?\(\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*(?:,\s*([\d.]+))?\s*\)')
_LENGTH_RE = re.compile(r'(-?[\d.]+)px')


def parse_color(value: str) -> tuple[int, int, int, float] | None:
    m = _COLOR_RE.match(value.strip())
    if not m:
        return None
    r, g, b, a = m.groups()
    alpha = float(a) if a is not None else 1.0
    return (int(r), int(g), int(b), alpha)


def colors_equal(a: str, b: str, tolerance: int = 12) -> bool:
    """Treats fully transparent colors as equal regardless of RGB channel
    values (a transparent black and a transparent white are visually
    identical) and otherwise compares channel-wise within `tolerance`."""
    ca, cb = parse_color(a), parse_color(b)
    if ca is None or cb is None:
        return a.strip() == b.strip()

    if ca[3] == 0 and cb[3] == 0:
        return True
    if abs(ca[3] - cb[3]) > 0.05:
        return False

    return all(abs(ca[i] - cb[i]) <= tolerance for i in range(3))


def parse_lengths(value: str) -> list[float]:
    """'8px 16px' -> [8.0, 16.0]; handles shorthand with 1-4 values."""
    return [float(x) for x in _LENGTH_RE.findall(value)]


def lengths_equal(a: str, b: str, tolerance_px: float = 2.0) -> bool:
    la, lb = parse_lengths(a), parse_lengths(b)

    if not la and not lb:
        return a.strip() == b.strip()  # e.g. both 'normal'
    if len(la) != len(lb):
        return False

    return all(abs(x - y) <= tolerance_px for x, y in zip(la, lb))


def style_value_equal(prop: str, a: str | None, b: str | None) -> bool:
    if a is None or b is None:
        return a == b

    if prop in COLOR_PROPS:
        return colors_equal(a, b)
    if prop in LENGTH_PROPS:
        return lengths_equal(a, b)

    return a.strip() == b.strip()


def style_match_rate(styles_a: dict, styles_b: dict) -> float:
    matches = sum(1 for p in STYLE_PROPS if style_value_equal(p, styles_a.get(p), styles_b.get(p)))
    return matches / len(STYLE_PROPS)

## 3. Element matching (bipartite / Hungarian)

Matches elements between the generated and reference snapshot. Cost combines
PrimeVue component identity (`data-pc-name`, falls back to tag when absent on
either side), text similarity, and normalized center distance. Pairs above
`MAX_MATCH_COST` are rejected even if Hungarian-optimal, to avoid forcing
nonsensical matches when element counts differ a lot.

In [13]:
def text_similarity(a: str, b: str) -> float:
    """Sorensen-Dice-like ratio via difflib -- 1.0 for identical text, 0.0
    for completely disjoint. Empty/empty counts as a perfect match (1.0)."""
    if not a and not b:
        return 1.0
    return difflib.SequenceMatcher(None, a, b).ratio()


def bbox_center(bbox: dict) -> tuple[float, float]:
    return bbox['x'] + bbox['width'] / 2, bbox['y'] + bbox['height'] / 2


def bbox_iou(a: dict, b: dict) -> float:
    ax1, ay1, ax2, ay2 = a['x'], a['y'], a['x'] + a['width'], a['y'] + a['height']
    bx1, by1, bx2, by2 = b['x'], b['y'], b['x'] + b['width'], b['y'] + b['height']

    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    intersection = iw * ih

    area_a = a['width'] * a['height']
    area_b = b['width'] * b['height']
    union = area_a + area_b - intersection

    return intersection / union if union > 0 else (1.0 if area_a == area_b == 0 else 0.0)


def match_cost(el_g: dict, el_r: dict, viewport: dict) -> float:
    pc_g, pc_r = el_g.get('pc_name'), el_r.get('pc_name')

    if pc_g or pc_r:
        component_mismatch = 0.0 if pc_g == pc_r else 1.0
    else:
        component_mismatch = 0.0 if el_g['tag'] == el_r['tag'] else 1.0

    text_dissim = 1.0 - text_similarity(el_g.get('text', ''), el_r.get('text', ''))

    cx_g, cy_g = bbox_center(el_g['bbox'])
    cx_r, cy_r = bbox_center(el_r['bbox'])
    vw, vh = max(viewport['width'], 1), max(viewport['height'], 1)
    position_distance = max(abs(cx_g - cx_r) / vw, abs(cy_g - cy_r) / vh)

    return (W_COMPONENT_MISMATCH * component_mismatch +
            W_TEXT_DISSIMILARITY * text_dissim +
            W_POSITION_DISTANCE * position_distance)


def match_elements(gen_snapshot: dict, ref_snapshot: dict) -> dict:
    """Returns matched pairs plus counts, ready for metric computation.

    Returns: {
        'matched': [(el_g, el_r, cost), ...],
        'tp': int, 'fp': int, 'fn': int,
    }
    """
    gen_elements = gen_snapshot.get('elements', [])
    ref_elements = ref_snapshot.get('elements', [])
    viewport = gen_snapshot.get('viewport', {'width': 1280, 'height': 800})

    if not gen_elements or not ref_elements:
        return {'matched': [], 'tp': 0, 'fp': len(gen_elements), 'fn': len(ref_elements)}

    cost_matrix = np.array([[match_cost(g, r, viewport) for r in ref_elements] for g in gen_elements])
    row_idx, col_idx = linear_sum_assignment(cost_matrix)

    matched = []
    matched_gen, matched_ref = set(), set()

    for gi, ri in zip(row_idx, col_idx):
        cost = cost_matrix[gi, ri]
        if cost <= MAX_MATCH_COST:
            matched.append((gen_elements[gi], ref_elements[ri], float(cost)))
            matched_gen.add(gi)
            matched_ref.add(ri)

    tp = len(matched)
    fp = len(gen_elements) - len(matched_gen)
    fn = len(ref_elements) - len(matched_ref)

    return {'matched': matched, 'tp': tp, 'fp': fp, 'fn': fn}

## 4. Per-mockup metrics

In [14]:
def position_similarity(bbox_g: dict, bbox_r: dict, viewport: dict) -> float:
    cx_g, cy_g = bbox_center(bbox_g)
    cx_r, cy_r = bbox_center(bbox_r)
    vw, vh = max(viewport['width'], 1), max(viewport['height'], 1)

    return 1.0 - max(abs(cx_g - cx_r) / vw, abs(cy_g - cy_r) / vh)


def size_similarity(bbox_g: dict, bbox_r: dict) -> float:
    def dim_sim(a, b):
        if a == 0 and b == 0:
            return 1.0
        return 1.0 - min(abs(a - b) / max(a, b, 1e-6), 1.0)

    return (dim_sim(bbox_g['width'], bbox_r['width']) + dim_sim(bbox_g['height'], bbox_r['height'])) / 2


def compute_visuel_fidelity_metrics(gen_snapshot: dict, ref_snapshot: dict) -> dict:
    """Note on text: element matching is deliberately lenient towards text
    differences (component identity dominates match cost, see MAX_MATCH_COST) --
    two Button instances with different labels can still be matched to each
    other rather than counted as FP+FN. That is the right choice for keeping
    position/size/style comparable even when a model mislabels a button, but
    it means Element-Match-Rate alone does NOT capture text correctness.
    mean_text_similarity is reported separately for exactly this reason --
    a high EMR with a low mean_text_similarity signals correctly-placed but
    mislabeled elements, which is itself a useful, distinct finding.
    """
    result = match_elements(gen_snapshot, ref_snapshot)
    matched = result['matched']
    tp, fp, fn = result['tp'], result['fp'], result['fn']

    element_match_rate = (2 * tp / (2 * tp + fp + fn)) if (2 * tp + fp + fn) > 0 else 1.0

    if not matched:
        return {
            'element_match_rate': element_match_rate, 'tp': tp, 'fp': fp, 'fn': fn,
            'mean_iou': None, 'mean_position_similarity': None,
            'mean_size_similarity': None, 'mean_style_match_rate': None,
            'mean_text_similarity': None,
        }

    viewport = gen_snapshot.get('viewport', {'width': 1280, 'height': 800})

    ious, pos_sims, size_sims, style_rates, text_sims = [], [], [], [], []
    for el_g, el_r, _cost in matched:
        ious.append(bbox_iou(el_g['bbox'], el_r['bbox']))
        pos_sims.append(position_similarity(el_g['bbox'], el_r['bbox'], viewport))
        size_sims.append(size_similarity(el_g['bbox'], el_r['bbox']))
        style_rates.append(style_match_rate(el_g.get('styles', {}), el_r.get('styles', {})))
        text_sims.append(text_similarity(el_g.get('text', ''), el_r.get('text', '')))

    return {
        'element_match_rate': round(element_match_rate, 4),
        'tp': tp, 'fp': fp, 'fn': fn,
        'mean_iou': round(sum(ious) / len(ious), 4),
        'mean_position_similarity': round(sum(pos_sims) / len(pos_sims), 4),
        'mean_size_similarity': round(sum(size_sims) / len(size_sims), 4),
        'mean_style_match_rate': round(sum(style_rates) / len(style_rates), 4),
        'mean_text_similarity': round(sum(text_sims) / len(text_sims), 4),
    }

## 5. File-naming parsing (identical to the other evaluation notebooks)

In [15]:
def parse_generated_stem(stem: str, approach: str) -> dict | None:
    parts = stem.split('-')

    if not parts[0].isdigit():
        return None

    index = parts[0].zfill(2)

    if approach == 'a':
        if len(parts) == 2 and parts[1] == 'a':
            return {'index': index, 'strategy': 'a', 'model': None, 'run': None}
        return None

    if len(parts) >= 4 and re.match(rf'^{approach}[123]$', parts[1]) and parts[-1].isdigit():
        strategy = parts[1]
        model = '-'.join(parts[2:-1])
        run = parts[-1]

        return {'index': index, 'strategy': strategy, 'model': model, 'run': run}

    return None


def gt_index(stem: str) -> str:
    return stem.zfill(2) if stem.isdigit() else stem

## 6. Main loop

In [16]:
uf3_results: list[dict] = []

for group in GROUP_LEVELS:
    # GT is variant-independent for 'uis' (flat, single set of snapshots
    # shared by both pretty and messy) but genuinely per-complexity for
    # 'components' -- look it up accordingly rather than assuming a
    # per-group GT set always exists.
    gt_group_key = GT_FLAT_GROUP_KEY if TYPE == 'uis' else group
    gt_snaps = SNAPSHOTS_BY_APPROACH['gt'].get(gt_group_key, {})
    gt_by_index = {gt_index(stem): snap for stem, snap in gt_snaps.items()}

    for approach in ['a', 'b', 'c', 'd']:
        approach_data = SNAPSHOTS_BY_APPROACH[approach]

        for prompt_strategy, by_group in approach_data.items():
            gen_snaps = by_group.get(group, {})

            for stem, gen_snapshot in gen_snaps.items():
                parsed = parse_generated_stem(stem, approach)

                if parsed is None:
                    print(f'  WARNING: file name does not match any pattern: '
                          f'{approach}/{prompt_strategy}/{group}/{stem}')
                    continue

                ref_snapshot = gt_by_index.get(parsed['index'])

                if ref_snapshot is None:
                    print(f'  MISSING GT snapshot for {approach}/{prompt_strategy}/{group}/{stem} '
                          f'(index {parsed["index"]})')
                    continue

                metrics = compute_visuel_fidelity_metrics(gen_snapshot, ref_snapshot)

                print(f'{group}/{parsed["index"]} '
                      f'[{approach}/{prompt_strategy or "-"}/{parsed["strategy"]}/{parsed["model"] or "-"}]  '
                      f'EMR={metrics["element_match_rate"]:.2f}  '
                      f'IoU={metrics["mean_iou"]}  Style={metrics["mean_style_match_rate"]}  '
                      f'Text={metrics["mean_text_similarity"]}')

                uf3_results.append({
                    'mockup':          f'{group}-{parsed["index"]}',
                    'complexity':       group,  # complexity for 'components', variant for 'uis'
                    'index':            parsed['index'],
                    'approach':         approach,
                    'strategy':         parsed['strategy'],
                    'prompt_strategy':  prompt_strategy,
                    'model':            parsed['model'] or '',
                    'run':              parsed['run'] or '',

                    'element_match_rate':       metrics['element_match_rate'],
                    'tp':                       metrics['tp'],
                    'fp':                       metrics['fp'],
                    'fn':                       metrics['fn'],
                    'mean_iou':                 metrics['mean_iou'],
                    'mean_position_similarity': metrics['mean_position_similarity'],
                    'mean_size_similarity':     metrics['mean_size_similarity'],
                    'mean_style_match_rate':    metrics['mean_style_match_rate'],
                    'mean_text_similarity':     metrics['mean_text_similarity'],
                })

print(f'\nComputed: {len(uf3_results)} mockup results')

simple/01 [a/-/a/-]  EMR=1.00  IoU=0.7066  Style=1.0  Text=1.0
simple/10 [a/-/a/-]  EMR=0.88  IoU=0.1907  Style=0.8571  Text=0.9688
simple/02 [a/-/a/-]  EMR=1.00  IoU=1.0  Style=1.0  Text=1.0
simple/03 [a/-/a/-]  EMR=1.00  IoU=0.1774  Style=0.9341  Text=1.0
simple/04 [a/-/a/-]  EMR=0.84  IoU=0.5903  Style=0.9864  Text=1.0
simple/05 [a/-/a/-]  EMR=0.84  IoU=0.1753  Style=0.9732  Text=1.0
simple/06 [a/-/a/-]  EMR=1.00  IoU=0.8033  Style=1.0  Text=1.0
simple/07 [a/-/a/-]  EMR=1.00  IoU=0.1808  Style=0.9881  Text=1.0
simple/08 [a/-/a/-]  EMR=0.62  IoU=0.399  Style=0.8929  Text=1.0
simple/09 [a/-/a/-]  EMR=0.10  IoU=0.0  Style=0.7857  Text=1.0
simple/01 [b/few_shot/b1/claude-sonnet-5]  EMR=0.95  IoU=0.2328  Style=0.9524  Text=1.0
simple/01 [b/few_shot/b1/gemini-3.1-pro-preview]  EMR=0.95  IoU=0.2328  Style=0.9762  Text=1.0
simple/01 [b/few_shot/b1/gpt-5.6-terra]  EMR=0.95  IoU=0.2328  Style=0.9683  Text=1.0
simple/01 [b/few_shot/b2/claude-sonnet-5]  EMR=0.95  IoU=0.2328  Style=0.9683  Text=

## 7. Aggregation (macro-/micro-averages)

In [17]:
def group_key(r: dict) -> tuple:
    return (r['approach'], r['strategy'], r['model'], r['prompt_strategy'])


def group_label(key: tuple) -> str:
    approach, strategy, model, prompt_strategy = key
    base = strategy if not model else f'{strategy}_{model}'

    # prompt_strategy is genuinely None for approach 'a' (and 'gt') -- omit
    # the suffix entirely instead of rendering Python's None as the literal
    # string 'None' in printed output.
    return f'{base}_{prompt_strategy}' if prompt_strategy else base


def macro_mean(items: list[dict], field: str) -> float | None:
    vals = [i[field] for i in items if i[field] is not None]
    return round(sum(vals) / len(vals), 4) if vals else None


def micro_element_match_rate(items: list[dict]) -> float | None:
    tp = sum(i['tp'] for i in items)
    fp = sum(i['fp'] for i in items)
    fn = sum(i['fn'] for i in items)
    denom = 2 * tp + fp + fn
    return round(2 * tp / denom, 4) if denom else None


def aggregate_uf3(items: list[dict]) -> dict:
    return {
        'n': len(items),
        'element_match_rate_macro': macro_mean(items, 'element_match_rate'),
        'element_match_rate_micro': micro_element_match_rate(items),
        'mean_iou_macro':                 macro_mean(items, 'mean_iou'),
        'mean_position_similarity_macro': macro_mean(items, 'mean_position_similarity'),
        'mean_size_similarity_macro':     macro_mean(items, 'mean_size_similarity'),
        'mean_style_match_rate_macro':    macro_mean(items, 'mean_style_match_rate'),
        'mean_text_similarity_macro':     macro_mean(items, 'mean_text_similarity'),
    }


by_method: dict[tuple, list[dict]] = defaultdict(list)
for r in uf3_results:
    by_method[group_key(r)].append(r)

by_method_complexity: dict[tuple, list[dict]] = defaultdict(list)
for r in uf3_results:
    by_method_complexity[(group_key(r), r['complexity'])].append(r)

GROUPS = sorted(by_method.keys(), key=lambda k: (k[0], k[1], k[2] or '', k[3]))

print(f'\n{"Method":42s} {"EMR":>6s} {"IoU":>6s} {"Pos":>6s} {"Size":>6s} {"Style":>6s} {"n":>4s}')
print('-' * 82)

for key in GROUPS:
    agg = aggregate_uf3(by_method[key])

    def fmt(v):
        return f'{v:.2f}' if v is not None else '-'

    print(f'{group_label(key):42s} {fmt(agg["element_match_rate_macro"]):>6s} '
          f'{fmt(agg["mean_iou_macro"]):>6s} {fmt(agg["mean_position_similarity_macro"]):>6s} '
          f'{fmt(agg["mean_size_similarity_macro"]):>6s} {fmt(agg["mean_style_match_rate_macro"]):>6s} '
          f'{agg["n"]:4d}')


Method                                        EMR    IoU    Pos   Size  Style    n
----------------------------------------------------------------------------------
a                                            0.78   0.60   0.90   0.83   0.96   30
b1_claude-sonnet-5_few_shot                  0.90   0.58   0.89   0.90   0.96   30
b1_claude-sonnet-5_zero_shot                 0.81   0.46   0.88   0.84   0.93   30
b1_gemini-3.1-pro-preview_few_shot           0.90   0.50   0.86   0.87   0.96   30
b1_gemini-3.1-pro-preview_zero_shot          0.79   0.39   0.85   0.80   0.93   30
b1_gpt-5.6-terra_few_shot                    0.80   0.43   0.84   0.82   0.93   30
b1_gpt-5.6-terra_zero_shot                   0.75   0.39   0.86   0.81   0.94   30
b2_claude-sonnet-5_few_shot                  0.91   0.56   0.89   0.91   0.96   30
b2_claude-sonnet-5_zero_shot                 0.86   0.45   0.84   0.86   0.95   30
b2_gemini-3.1-pro-preview_few_shot           0.86   0.47   0.85   0.82   0.95   30
b2_

## 8. CSV export

In [18]:
EVALUATIONS_DIR = Path(f'evaluations/{TYPE}')
EVALUATIONS_DIR.mkdir(parents=True, exist_ok=True)

# 1) Long format
BY_MOCKUP_CSV = EVALUATIONS_DIR / f'eval_visual_fidelity_{TYPE}_by_mockup.csv'
BY_MOCKUP_FIELDNAMES = [
    'mockup', 'complexity', 'index', 'approach', 'strategy', 'prompt_strategy', 'model', 'run',
    'element_match_rate', 'tp', 'fp', 'fn',
    'mean_iou', 'mean_position_similarity', 'mean_size_similarity', 'mean_style_match_rate',
    'mean_text_similarity',
]

with open(BY_MOCKUP_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=BY_MOCKUP_FIELDNAMES)
    writer.writeheader()
    writer.writerows(uf3_results)

print(f'Saved: {BY_MOCKUP_CSV}  ({len(uf3_results)} rows)')

# 2) Summary per configuration
SUMMARY_CSV = EVALUATIONS_DIR / f'eval_visual_fidelity_{TYPE}_summary.csv'
SUMMARY_FIELDNAMES = [
    'approach', 'strategy', 'prompt_strategy', 'model', 'n',
    'element_match_rate_macro', 'element_match_rate_micro',
    'mean_iou_macro', 'mean_position_similarity_macro',
    'mean_size_similarity_macro', 'mean_style_match_rate_macro', 'mean_text_similarity_macro',
]

summary_rows = []
for key in GROUPS:
    approach, strategy, model, prompt_strategy = key
    agg = aggregate_uf3(by_method[key])
    summary_rows.append({
        'approach': approach, 'strategy': strategy,
        'prompt_strategy': prompt_strategy, 'model': model or '',
        **agg,
    })

with open(SUMMARY_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=SUMMARY_FIELDNAMES)
    writer.writeheader()
    writer.writerows(summary_rows)

print(f'Saved: {SUMMARY_CSV}  ({len(summary_rows)} rows)')

# 3) Per configuration x complexity
BY_COMPLEXITY_CSV = EVALUATIONS_DIR / f'eval_visual_fidelity_{TYPE}_by_complexity.csv'
BY_COMPLEXITY_FIELDNAMES = ['approach', 'strategy', 'prompt_strategy', 'model', 'complexity'] + \
    [f for f in SUMMARY_FIELDNAMES if f not in ('approach', 'strategy', 'prompt_strategy', 'model')]

by_complexity_rows = []
for key in GROUPS:
    approach, strategy, model, prompt_strategy = key

    for group in GROUP_LEVELS:
        items = by_method_complexity[(key, group)]
        if not items:
            continue

        agg = aggregate_uf3(items)
        by_complexity_rows.append({
            'approach': approach, 'strategy': strategy, 'prompt_strategy': prompt_strategy,
            'model': model or '', 'complexity': group, **agg,  # complexity for 'components', variant for 'uis'
        })

with open(BY_COMPLEXITY_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=BY_COMPLEXITY_FIELDNAMES)
    writer.writeheader()
    writer.writerows(by_complexity_rows)

print(f'Saved: {BY_COMPLEXITY_CSV}  ({len(by_complexity_rows)} rows)')

# 4a) Degradation factor (medium/simple, hard/simple) per configuration -- on
#     Element-Match-Rate. 'components' only, since 'simple'/'medium'/'hard'
#     is an ordered complexity progression that does not apply to 'uis'.
def degradation_factor(base, target):
    if base is None or target is None or base == 0:
        return None
    return round(target / base, 4)


if TYPE == 'components':
    DEGRADATION_CSV = EVALUATIONS_DIR / f'eval_visual_fidelity_{TYPE}_degradation.csv'
    DEGRADATION_FIELDNAMES = [
        'approach', 'strategy', 'prompt_strategy', 'model',
        'emr_simple', 'emr_medium', 'emr_hard',
        'degradation_emr_medium', 'degradation_emr_hard',
    ]

    degradation_rows = []
    for key in GROUPS:
        approach, strategy, model, prompt_strategy = key

        aggs = {c: (aggregate_uf3(by_method_complexity[(key, c)]) if by_method_complexity[(key, c)] else None)
                for c in COMPLEXITIES}
        emr = {c: (aggs[c]['element_match_rate_macro'] if aggs[c] else None) for c in COMPLEXITIES}

        degradation_rows.append({
            'approach': approach, 'strategy': strategy, 'prompt_strategy': prompt_strategy, 'model': model or '',
            'emr_simple': emr['simple'], 'emr_medium': emr['medium'], 'emr_hard': emr['hard'],
            'degradation_emr_medium': degradation_factor(emr['simple'], emr['medium']),
            'degradation_emr_hard':   degradation_factor(emr['simple'], emr['hard']),
        })

    with open(DEGRADATION_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=DEGRADATION_FIELDNAMES)
        writer.writeheader()
        writer.writerows(degradation_rows)

    print(f'Saved: {DEGRADATION_CSV}  ({len(degradation_rows)} rows)')

else:
    # 4b) Pretty/Messy robustness ratio per configuration -- the UF5
    #     Robustheitsindex R(M,P) = EMR_messy / EMR_pretty, the 'uis'
    #     equivalent of the complexity degradation factor above.
    ROBUSTNESS_CSV = EVALUATIONS_DIR / f'eval_visual_fidelity_{TYPE}_robustness.csv'
    ROBUSTNESS_FIELDNAMES = [
        'approach', 'strategy', 'prompt_strategy', 'model',
        'emr_pretty', 'emr_messy', 'robustness_index',
    ]

    robustness_rows = []
    for key in GROUPS:
        approach, strategy, model, prompt_strategy = key

        aggs = {v: (aggregate_uf3(by_method_complexity[(key, v)]) if by_method_complexity[(key, v)] else None)
                for v in VARIANTS}
        emr = {v: (aggs[v]['element_match_rate_macro'] if aggs[v] else None) for v in VARIANTS}

        robustness_rows.append({
            'approach': approach, 'strategy': strategy, 'prompt_strategy': prompt_strategy, 'model': model or '',
            'emr_pretty': emr['pretty'], 'emr_messy': emr['messy'],
            'robustness_index': degradation_factor(emr['pretty'], emr['messy']),
        })

    with open(ROBUSTNESS_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=ROBUSTNESS_FIELDNAMES)
        writer.writeheader()
        writer.writerows(robustness_rows)

    print(f'Saved: {ROBUSTNESS_CSV}  ({len(robustness_rows)} rows)')

Saved: evaluations\components\eval_visual_fidelity_components_by_mockup.csv  (1650 rows)
Saved: evaluations\components\eval_visual_fidelity_components_summary.csv  (55 rows)
Saved: evaluations\components\eval_visual_fidelity_components_by_complexity.csv  (165 rows)
Saved: evaluations\components\eval_visual_fidelity_components_degradation.csv  (55 rows)
